# Training Faster R-CNN with k-fold cross validation

The code in here is roughly based on this tutorial
https://pytorch.org/tutorials/intermediate/torchvision_tutorial.html


## Testing `forward()` method

Before iterating over the dataset, it's good to see what the model expects
during training and inference time on sample data.


In [ ]:
from dataset_transforms import get_transforms
from model import get_model_instance_segmentation
import utils
from instance_dataset import BoundingBoxDataset

import torch


model = get_model_instance_segmentation(BoundingBoxDataset.num_classes)

dataset = BoundingBoxDataset("../../training_data", get_transforms(is_train=True))
data_loader = torch.utils.data.DataLoader(
    dataset, batch_size=2, shuffle=True, collate_fn=utils.collate_fn
)

# For Training
images, targets = next(iter(data_loader))
images = list(image for image in images)
targets = [{k: v for k, v in t.items()} for t in targets]
output = model(images, targets)  # Returns losses and detections
print(output)

# For inference
model.eval()
x = [torch.rand(3, 300, 400), torch.rand(3, 500, 400)]
predictions = model(x)  # Returns predictions
print(predictions[0])

## Training code


In [ ]:
import csv
from engine import evaluate, train_one_epoch
import time

from torch.optim import AdamW


def setup_csv(file_name):
    """
    Set up the CSV file with the appropriate headers.
    """
    with open(file_name, "w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(
            [
                "fold",
                "epoch",
                "train_loss",
                "train_loss_classifier",
                "train_loss_box_reg",
                "train_loss_objectness",
                "train_loss_rpn_box_reg",
                "val_loss",
                "val_loss_classifier",
                "val_loss_box_reg",
                "val_loss_objectness",
                "val_loss_rpn_box_reg",
                "val_avg_precision_iou_50",
                "val_avg_recall_iou_50",
                "val_avg_absolute_error",
                "val_avg_relative_error",
            ]
        )

    def append_to_csv(row):
        """
        Append a row of data to the CSV file.
        """
        with open(file_name, "a", newline="") as file:
            writer = csv.writer(file)
            writer.writerow(row)

    return append_to_csv


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} for training.")


def train_model(train_set, val_set, num_epochs, save_scores=None, fold=None):
    # define training and validation data loaders
    train_loader = torch.utils.data.DataLoader(
        train_set, batch_size=8, shuffle=True, collate_fn=utils.collate_fn
    )

    if val_set is not None:
        val_loader = torch.utils.data.DataLoader(
            val_set, batch_size=8, shuffle=False, collate_fn=utils.collate_fn
        )

    model = get_model_instance_segmentation(BoundingBoxDataset.num_classes).to(device)

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = AdamW(params, lr=0.001, weight_decay=0.01)

    for epoch in range(num_epochs):
        # train for one epoch, printing every 10 iterations
        train_losses = train_one_epoch(
            model, optimizer, train_loader, device, epoch, print_freq=10
        )

        avg_train_loss_classifier = sum(
            [loss["loss_classifier"] for loss in train_losses]
        ) / len(train_losses)
        avg_train_loss_box_reg = sum(
            [loss["loss_box_reg"] for loss in train_losses]
        ) / len(train_losses)
        avg_train_loss_objectness = sum(
            [loss["loss_objectness"] for loss in train_losses]
        ) / len(train_losses)
        avg_train_loss_rpn_box_reg = sum(
            [loss["loss_rpn_box_reg"] for loss in train_losses]
        ) / len(train_losses)
        avg_train_loss = sum(
            [
                (
                    loss["loss_classifier"]
                    + loss["loss_box_reg"]
                    + loss["loss_objectness"]
                    + loss["loss_rpn_box_reg"]
                )
                for loss in train_losses
            ]
        ) / len(train_losses)

        # check loss on validation set
        if val_set is not None:
            with torch.no_grad():
                val_losses = train_one_epoch(
                    model,
                    optimizer,
                    val_loader,
                    device,
                    epoch,
                    print_freq=10,
                    optimize=False,
                )

                avg_val_loss_classifier = sum(
                    [loss["loss_classifier"] for loss in val_losses]
                ) / len(val_losses)
                avg_val_loss_box_reg = sum(
                    [loss["loss_box_reg"] for loss in val_losses]
                ) / len(val_losses)
                avg_val_loss_objectness = sum(
                    [loss["loss_objectness"] for loss in val_losses]
                ) / len(val_losses)
                avg_val_loss_rpn_box_reg = sum(
                    [loss["loss_rpn_box_reg"] for loss in val_losses]
                ) / len(val_losses)
                avg_val_loss = sum(
                    [
                        (
                            loss["loss_classifier"]
                            + loss["loss_box_reg"]
                            + loss["loss_objectness"]
                            + loss["loss_rpn_box_reg"]
                        )
                        for loss in val_losses
                    ]
                ) / len(val_losses)

                # evaluate on the validation dataset
                coco_evaluator, avg_abs_error, avg_rel_error = evaluate(
                    model, val_loader, device=device
                )

                stats = coco_evaluator.coco_eval["bbox"].stats

                avg_precision_iou_50 = stats[0]
                avg_recall_iou_50 = stats[8]

        if save_scores is not None:
            save_scores(
                [
                    float(fold),
                    float(epoch),
                    float(avg_train_loss),
                    float(avg_train_loss_classifier),
                    float(avg_train_loss_box_reg),
                    float(avg_train_loss_objectness),
                    float(avg_train_loss_rpn_box_reg),
                    float(avg_val_loss),
                    float(avg_val_loss_classifier),
                    float(avg_val_loss_box_reg),
                    float(avg_val_loss_objectness),
                    float(avg_val_loss_rpn_box_reg),
                    float(avg_precision_iou_50),
                    float(avg_recall_iou_50),
                    float(avg_abs_error),
                    float(avg_rel_error),
                ],
            )

    return model

## K-fold cross validation

K different models are trained and validated here


In [ ]:
from sklearn.model_selection import KFold

k_folds = 5
num_epochs = 50
kfold = KFold(n_splits=k_folds, shuffle=True)

# use our dataset and defined transformations
full_dataset = BoundingBoxDataset("../../training_data", get_transforms(is_train=True))

write_row = setup_csv(f"validation_results_{time.time()}.csv")

for fold, (train_ids, val_ids) in enumerate(kfold.split(full_dataset)):
    print(f"FOLD {fold}")
    print("-" * 20)

    train_dataset = torch.utils.data.Subset(full_dataset, train_ids)
    train_dataset.dataset.transforms = get_transforms(is_train=True)

    val_dataset = torch.utils.data.Subset(full_dataset, val_ids)
    val_dataset.dataset.transforms = get_transforms(is_train=False)

    model = train_model(
        train_dataset,
        val_dataset,
        num_epochs,
        save_scores=write_row,
        fold=fold,
    )

    torch.save(model.state_dict(), f"model_fold_{fold}.pth")

## Final training run

The final training run on all of the available training data. Model is also
saved to a file.


In [ ]:
import time

print(f"FINAL TRAINING")
print("-" * 20)

write_row = setup_csv(f"training_results_{time.time()}.csv")

final_model = train_model(
    full_dataset,
    num_epochs=num_epochs,
    save_scores=write_row,
)

# save the model
torch.save(
    final_model.state_dict(),
    f"faster-rcnn-{num_epochs}-epochs-{time.time()}.pth",
)